# Python Bridge for 08a

## A Short Code Refresher

Use this optional notebook alongside 08a for a closer look at the Python mechanics of DBSCAN and Gaussian mixtures. Plan for about 20 minutes. All the small example tables are supplied below.

The lecture develops the methods and the evidence needed to interpret clusters. This companion follows five code patterns:

1. fit DBSCAN and attach its labels;
2. distinguish core, border, and noise observations;
3. use a Boolean mask to summarize clustered rows;
4. inspect a fitted Gaussian mixture; and
5. compare hard labels with membership probabilities, including for new rows.

## References

The scikit-learn references for [DBSCAN](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.DBSCAN.html) and [GaussianMixture](https://scikit-learn.org/stable/modules/generated/sklearn.mixture.GaussianMixture.html) describe the fitted attributes and prediction methods used here.

## Create a Small Coordinate Table

Each row is one made-up point. Both coordinates use the same units, so the example uses Euclidean distance directly. In a problem with differently scaled features, choose the representation and any scaling before selecting DBSCAN's radius.

In [ ]:
import pandas as pd
from sklearn.cluster import DBSCAN
from sklearn.mixture import GaussianMixture

In [ ]:
points = pd.DataFrame(
    {
        "x": [0, 0, 0.2, 0.4, 3, 3, 3.2, 3.4, 1.5],
        "y": [0, 0.2, 0, 0, 3, 3.2, 3, 3, 1.5],
    },
    index=list("ABCDEFGHI"),
)

print("input shape:", points.shape)
points

There are nine observations with two coordinates each. A–D lie near the origin, E–H lie near (3, 3), and I lies between them.

## Fit DBSCAN and Read the Labels

`eps=0.25` is the neighborhood radius in these coordinate units. `min_samples=3` requires at least three observations within that radius, counting the observation itself, for a point to be core. `.fit()` stores the result on the model.

In [ ]:
density_model = DBSCAN(eps=0.25, min_samples=3)
density_model.fit(points)
density_labels = pd.Series(density_model.labels_, index=points.index, name="cluster")

points.assign(cluster=density_labels)

A–D receive one cluster label and E–H another. I receives `-1`, scikit-learn's noise label. Nonnegative labels identify clusters; they do not rank them. Noise means unassigned under these settings, not proof that the observation is an error.

## Locate Core and Border Points

`core_sample_indices_` contains integer positions in the fitting input. Build a Boolean Series with the original row labels, then use `.iloc` to mark those positions. `.loc` would instead interpret integers as index labels.

In [ ]:
is_core = pd.Series(False, index=points.index, name="Core")
is_core.iloc[density_model.core_sample_indices_] = True
is_noise = density_labels.eq(-1)
is_border = density_labels.ge(0) & ~is_core

status = pd.DataFrame(
    {"Cluster": density_labels, "Core": is_core, "Border": is_border, "Noise": is_noise}
)
status

A and C are core in the first cluster; E and G are core in the second. B, D, F, and H are border points: they join a cluster through a nearby core point but do not have enough neighbors to be core themselves. I is noise. `~` negates a Boolean Series, and `&` requires both conditions to hold.

## Summarize Clusters Without Treating Noise as One

A Boolean mask selects the rows with nonnegative labels. Count noise separately, since all rows labeled `-1` need not form a coherent group.

In [ ]:
assigned = density_labels.ge(0)
clustered = points.loc[assigned].assign(cluster=density_labels.loc[assigned])
cluster_sizes = clustered.groupby("cluster").size()

print("cluster sizes:")
print(cluster_sizes)
print("noise observations:", int(is_noise.sum()))
print("number of clusters:", density_labels.loc[assigned].nunique())

The result has two clusters of four observations and one noise observation. The mask and point table share the same index, so `.loc` retains the correct rows. scikit-learn's DBSCAN has no `.predict()` method for assigning new observations to a fitted result. Adding rows and fitting again can change which existing observations are core, border, or noise.

## Prepare a Separate Mixture Example

For the mixture workflow, use the eight listed observations A–H. This is a deliberately selected code example. Removing DBSCAN noise is not a required preprocessing step for a Gaussian mixture; the analytical question should determine which observations belong in a fit.

In [ ]:
mixture_points = points.loc[list("ABCDEFGH")].copy()
mixture = GaussianMixture(n_components=2, covariance_type="full", n_init=10, random_state=7130)
mixture.fit(mixture_points)

print("converged:", mixture.converged_)
print("mean-array shape:", mixture.means_.shape)
print("covariance-array shape:", mixture.covariances_.shape)

`n_components=2` requests two Gaussian components. `covariance_type="full"` allows each its own spread and orientation, as in 08a. `n_init=10` tries ten initializations, and the fixed seed makes the random choices repeatable. The means have shape `(2, 2)`; full covariances have shape `(2, 2, 2)`, one two-by-two matrix per component. Check `converged_` before interpreting the fit. Eight observations suffice for inspecting these objects, but provide little evidence about real population shapes.

## Inspect Component Summaries

Use one row per component when displaying the learned means and weights. Each weight is that component's estimated share of the fitted mixture.

In [ ]:
component_table = pd.DataFrame(mixture.means_, columns=mixture_points.columns).rename_axis(
    "Component"
)
component_table["Weight"] = mixture.weights_

component_table.round(3)

The fitted means are approximately (0.15, 0.05) and (3.15, 3.05), each with weight 0.5. Component numbering is arbitrary. Its meaning comes from the corresponding means, covariance, and membership column, not from the numeral itself.

## Compare Labels and Probabilities

`predict_proba()` returns one column per component. Each row gives the observation's relative membership probabilities under this fitted mixture. `predict()` selects the component with the largest probability.

In [ ]:
probabilities = mixture.predict_proba(mixture_points)
hard_labels = mixture.predict(mixture_points)
probability_columns = [f"P(component {i})" for i in range(mixture.n_components)]
membership = pd.DataFrame(probabilities, index=mixture_points.index, columns=probability_columns)
membership["Hard label"] = hard_labels

print("probability shape:", probabilities.shape)
print("label shape:", hard_labels.shape)
print("probability row sums:", probabilities.sum(axis=1).round(6))
membership.round(3)

The shapes are `(8, 2)` and `(8,)`, and each probability row sums to 1. `axis=1` means add across columns within each row. These separated points have probabilities close to 0 or 1. Unlike DBSCAN, the mixture assigns every supplied row to a component; it has no special noise label.

## Inspect New Observations Without Refitting

One new point lies near the first group and one lies midway between the two fitted means. They use the same coordinates and units as the fitting input. With standardized features, you would first apply the existing fitted scaler.

In [ ]:
new_points = pd.DataFrame({"x": [0.1, 1.65], "y": [0.05, 1.55]}, index=["Near A", "Midway"])
new_probabilities = mixture.predict_proba(new_points)
new_membership = pd.DataFrame(
    new_probabilities, index=new_points.index, columns=probability_columns
)
new_membership["Hard label"] = mixture.predict(new_points)

new_membership.round(3)

The first row strongly favors the nearby component. The midpoint gives about 0.5 to each, yet `predict()` still returns one label. That label hides the ambiguity. The probabilities compare components within this fitted model; they do not establish that a point is typical or that either component is a real population. Neither prediction call changes the learned parameters.

## Try One Small Modification

Fit a separate DBSCAN object with `min_samples=4` and the same radius. Predict the result by counting neighbors in the small point table before running the cell.

In [ ]:
stricter_model = DBSCAN(eps=0.25, min_samples=4).fit(points)

points.assign(original=density_labels, stricter=stricter_model.labels_)

Every observation is now noise because no point has four observations within radius 0.25, including itself. Without a core point, nearby non-core points cannot form a DBSCAN cluster. The original `density_model` and fitted mixture remain available unchanged.

## Ready for 08a

You are ready to return to 08a when you can recognize these patterns:

```text
DBSCAN(...).fit(points)           # store density-based labels and core positions
model.labels_ == -1              # identify DBSCAN noise
model.core_sample_indices_       # locate core rows by integer position
GaussianMixture(...).fit(points)  # learn component parameters
mixture.predict_proba(points)    # retain every component's membership probability
mixture.predict(points)          # select one component label per row
```

Keep labels attached to their input rows. A DBSCAN noise flag, a Gaussian component label, and a membership probability answer different questions, even when all three are displayed beside the same observation.

---

Auburn University / Industrial and Systems Engineering<br>
INSY 7130, Pattern Discovery and Time Series Analysis<br>
© Copyright Danny J. O'Leary.

For course materials, attribution, and licensing information, see the [INSY 7130 course-materials README](../../README.md).